# LangChain In Depth — Concepts + Interview Prep

`16Langchain-Core.ipynb` covered exactly what Day 16 asked for (LCEL, loaders,
splitters, a tiny RAG). This notebook goes wider — the rest of what LangChain
actually is, and how to talk about it in an interview. Every mechanism below is
demoed with real, executed code, not just described.

**How to use this:** each section has a short demo, then an **"Interview
angle"** callout — the actual sentence-level answer you'd give if asked about
it live. Section 11 is a consolidated Q&A cheat sheet you can skim the night
before an interview.

In [1]:
from dotenv import load_dotenv
load_dotenv()
print("Environment loaded.")

Environment loaded.


## 1. What LangChain actually is (the elevator pitch)

LangChain is an **orchestration framework**, not a model and not a vector
database. It doesn't do anything an LLM API call, a vector DB client, and some
Python glue code couldn't do by hand — you proved that already in
`13Embeddings.ipynb` and `14ChromaDb.py`, which used zero LangChain.

What it actually gives you:
- **A common interface** across ~700 integrations (every LLM provider, vector
  store, document source) — swap Chroma for Pinecone without rewriting your
  pipeline logic.
- **LCEL** — a composition syntax (`|`) for chaining those pieces together,
  with batching/streaming/async for free on anything built with it.
- **Prebuilt patterns** for the recurring hard parts: RAG chains, agents,
  structured output, memory.

> **Interview angle:** "LangChain is a framework for composing LLM
> applications out of interchangeable pieces — models, retrievers, tools,
> parsers — using a consistent interface (Runnables) and a composition syntax
> (LCEL). The value isn't capability you couldn't build yourself; it's fewer
> integration points to hand-maintain and swap between providers."


## 2. LCEL & the Runnable interface

Every LangChain component — a prompt, a model, a retriever, a parser, even a
plain Python function — is a **Runnable**. That's *why* the `|` operator works
across all of them: `|` is just `RunnableSequence` under the hood, and every
Runnable implements the same 4 methods: `.invoke()`, `.batch()`, `.stream()`,
`.ainvoke()` (async). Learn one, get all four on everything you build.

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template("Say hi to {name} in 3 words.")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = prompt | llm | (lambda r: r.content)

# .invoke() -- one input, one output (what you've used so far)
print("invoke:", chain.invoke({"name": "Alice"}))

# .batch() -- multiple inputs, run concurrently, same chain, zero extra code
print("batch: ", chain.batch([{"name": "Alice"}, {"name": "Bob"}]))

# .stream() -- tokens as they arrive, instead of waiting for the full response
print("stream: ", end="")
for chunk in llm.stream("Count from 1 to 5."):
    print(chunk.content, end="", flush=True)
print()

invoke: Hello, Alice! 😊


batch:  ['Hello, Alice! 😊', 'Hello, Bob! Howdy!']
stream: 

1

,

2

...

,

4

,

5

.

**`RunnableParallel` and `RunnableLambda`** are the two building blocks behind
that `{"context": ..., "question": ...}` dict pattern from the RAG chain in
`16Langchain-Core.ipynb` — a dict of Runnables *is* a `RunnableParallel`
(run every branch on the same input, concurrently), and any plain Python
function becomes a Runnable by wrapping it in `RunnableLambda` (or LangChain
does it for you automatically, which is what happened with `format_docs`).

In [3]:
from langchain_core.runnables import RunnableLambda

count_words = RunnableLambda(lambda text: len(text.split()))
print("word count:", count_words.invoke("this sentence has five words"))

word count: 5


> **Interview angle:** "Every LangChain component implements the Runnable
> interface — `invoke`, `batch`, `stream`, `ainvoke`. LCEL's `|` builds a
> `RunnableSequence`; a dict of Runnables becomes a `RunnableParallel` that
> fans out to every branch on the same input. That's why the same 4 methods
> work identically whether you're calling one prompt or a 10-step chain."


## 3. Prompts — beyond a single template

Two patterns come up constantly beyond the basic `ChatPromptTemplate` you've
already used: **few-shot prompting** (show examples instead of just
instructions) and **chat history** (multi-turn conversations, not one-shot
questions).

In [4]:
from langchain_core.prompts import MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder("history"),   # <-- prior turns get inserted here
    ("human", "{input}"),
])

history_chain = chat_prompt | llm | (lambda r: r.content)

result = history_chain.invoke({
    "history": [("human", "My name is Yogesh."), ("ai", "Nice to meet you, Yogesh!")],
    "input": "What is my name?",
})
print(result)

Your name is Yogesh.


Notice: the model only "remembers" your name because the full history was
re-sent as part of the prompt — LLMs are stateless between calls. Every
"memory" mechanism in LangChain (and every chatbot you've ever used) is really
just re-sending prior turns as context, not the model actually retaining
anything.

> **Interview angle:** "LLMs have no memory between calls — every apparent
> 'memory' is the application re-sending prior conversation turns as part of
> the prompt. `MessagesPlaceholder` is where LangChain injects that history
> into a `ChatPromptTemplate`. This matters for cost/context-window planning:
> a long conversation means a growing prompt, not a growing model state."


## 4. Structured output — the modern way

You already know this exact problem from `04email_extractor.py` — get the LLM
to return real Pydantic objects, not a string you have to parse yourself.
LangChain's current recommended approach is `.with_structured_output()`
(older code you'll see in tutorials uses `PydanticOutputParser` — same idea,
more manual, mostly superseded now).

In [5]:
from pydantic import BaseModel, Field

class FilingSummary(BaseModel):
    company: str = Field(description="company name")
    risk_level: str = Field(description="low, medium, or high")

structured_llm = llm.with_structured_output(FilingSummary)
result = structured_llm.invoke("Apple faces moderate competitive and regulatory risks.")
print(result)
print(type(result))

company='Apple' risk_level='medium'
<class '__main__.FilingSummary'>


Same underlying idea as `instructor` (Days 4-5's Email Classifier) — both give
you a validated Pydantic object back instead of raw text. `instructor` works
directly against provider SDKs (Gemini, OpenAI, Anthropic) with a lighter
footprint; `with_structured_output` is the same pattern built into LangChain
so it composes with `|` like everything else.

> **Interview angle:** "`with_structured_output` binds a Pydantic schema to
> the model call and validates the response automatically — same underlying
> mechanism as tool-calling/JSON mode under the hood, just wrapped so it fits
> into an LCEL chain. I'd reach for this over manually parsing JSON any time
> the output feeds into code rather than a human."


## 5. Document loaders & text splitters — the other types

`16Langchain-Core.ipynb` covered `PyPDFLoader` + `RecursiveCharacterTextSplitter`
— the two you'll use most for Project 1. Worth knowing the landscape exists
beyond those two, since interviewers often probe "what would you use for X":

| Loader | Use case |
|---|---|
| `PyPDFLoader` | PDFs (what you're using) |
| `WebBaseLoader` | Scrape a webpage into a Document |
| `CSVLoader` | One Document per row |
| `DirectoryLoader` | Load every file in a folder at once |

| Splitter | Use case |
|---|---|
| `CharacterTextSplitter` | Naive, splits on one fixed separator only |
| `RecursiveCharacterTextSplitter` | What you're using — tries paragraph → sentence → word boundaries, in order, to avoid mid-sentence cuts |
| `TokenTextSplitter` | Splits by model token count, not character count — matters because token count is what actually determines context-window/cost limits, not character count |
| `MarkdownHeaderTextSplitter` | Splits on `#`/`##` headers, keeps section structure as metadata |
| Semantic chunking (`SemanticChunker`, LangChain Experimental) | Splits where embedding similarity between sentences drops — chunk boundaries follow *meaning* shifts, not a fixed size. Slower and costs embedding calls just to split, but higher quality on documents with uneven paragraph lengths. |

> **Interview angle:** "`RecursiveCharacterTextSplitter` is the default because
> it degrades gracefully — tries to split on paragraph breaks first, then
> sentences, then words, rather than cutting at a hard character count
> mid-word. For a real 10-K I'd also consider `MarkdownHeaderTextSplitter`-style
> section awareness, since SEC filings have a strict `Item 1`, `Item 1A`
> structure worth preserving as retrievable metadata."


## 6. Retrievers — similarity isn't the only option

`16Langchain-Core.ipynb` used the default retriever (plain cosine similarity —
same math as Day 13). There's a real, common problem with plain similarity
search: the top-k results can be near-duplicates of each other, wasting your
context budget on redundant information instead of covering diverse angles of
the question.

**MMR (Maximal Marginal Relevance)** fixes this — it still ranks by
similarity, but also actively penalizes picking a result too similar to one
already selected.

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

docs = [
    Document(page_content="Apple reported strong iPhone revenue growth this quarter."),
    Document(page_content="Apple faces supply chain risk from concentrated manufacturing in Asia."),
    Document(page_content="It rained heavily in Seattle yesterday."),
]
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vs = Chroma.from_documents(docs, embeddings, collection_name="mmr_demo")

mmr_retriever = vs.as_retriever(search_type="mmr", search_kwargs={"k": 2, "fetch_k": 3})
for r in mmr_retriever.invoke("How is Apple performing financially?"):
    print("-", r.page_content)

- Apple reported strong iPhone revenue growth this quarter.
- Apple faces supply chain risk from concentrated manufacturing in Asia.


Other retriever variants worth knowing by name (not all demoed here, since
they mostly change *what feeds the vector search*, not the mechanism itself):

- **Multi-query retriever** — has the LLM generate several rephrasings of the
  user's question first, retrieves for each, merges results. Helps when the
  user's exact wording doesn't match the document's wording well.
- **Self-query retriever** — the LLM converts a natural-language question into
  a structured filter (e.g. "risks mentioned in 2024" → metadata filter
  `year=2024` + semantic search on "risks") — combines metadata filtering with
  semantic search.
- **Ensemble retriever** — combines a keyword search (BM25) with vector
  search, since embeddings can miss exact-term matches (ticker symbols, legal
  citations, specific numbers) that keyword search catches trivially. This is
  what people mean by **hybrid search**.

> **Interview angle:** "Plain similarity search can return near-duplicate
> chunks, wasting context budget. MMR trades a bit of top-1 relevance for
> diversity across the retrieved set. For production RAG over financial
> filings specifically, I'd lean toward hybrid search — vector search alone
> misses exact-match cases like ticker symbols or specific dollar figures that
> BM25/keyword search catches natively."


## 7. Tools & Agents — when a chain isn't enough

Everything so far has been a **chain**: a fixed sequence of steps, decided by
you, at code-writing time. A **tool** is a Python function the LLM can choose
to call; an **agent** is a loop where the LLM decides *which* tools to call,
*in what order*, based on what it's seen so far — the control flow is decided
by the model at run time, not hard-coded by you.

In [7]:
from langchain_core.tools import tool

@tool
def get_stock_price(ticker: str) -> str:
    '''Look up the current stock price for a ticker symbol.'''
    prices = {"AAPL": 262.50, "MSFT": 510.10, "JPM": 305.20}
    return f"${prices.get(ticker.upper(), 0):.2f}"

print("Tool name:  ", get_stock_price.name)
print("Tool schema:", get_stock_price.args)

# bind_tools: the LLM sees the tool's schema and CAN choose to call it --
# it doesn't call it itself, it just tells you it wants to
llm_with_tools = llm.bind_tools([get_stock_price])
response = llm_with_tools.invoke("What is the price of AAPL stock?")
print("\nModel wants to call:", response.tool_calls)

Tool name:   get_stock_price
Tool schema: {'ticker': {'title': 'Ticker', 'type': 'string'}}



Model wants to call: [{'name': 'get_stock_price', 'args': {'ticker': 'AAPL'}, 'id': 'call_Iw5sDJVcaXGYl6VK8RSOY7d8', 'type': 'tool_call'}]


Notice `llm_with_tools.invoke(...)` didn't actually run the function — it
returned a `tool_calls` list saying *which* tool it wants called with *which*
arguments. Actually executing it and feeding the result back is the "agent
loop" part. `create_agent` (from LangGraph, current recommended path) wraps
that whole loop for you.

In [8]:
from langchain.agents import create_agent

agent = create_agent(llm, [get_stock_price])
result = agent.invoke({"messages": [{"role": "user", "content": "What is AAPL trading at?"}]})
print(result["messages"][-1].content)

AAPL is currently trading at $262.50.


This is the **ReAct pattern** (Reason + Act): the model reasons about what it
needs, picks a tool, observes the result, and repeats until it has enough to
answer — all inside that one `agent.invoke()` call.

> **Interview angle:** "A chain has control flow decided at code-writing time
> — the same steps run every time. An agent has the LLM decide control flow
> at run time — which tools to call, in what order, whether to call another
> tool based on the first result. I'd default to a chain whenever the steps
> are actually fixed — it's cheaper, faster, and fully predictable. Agents
> are for genuinely open-ended tasks where you can't enumerate the steps in
> advance."


## 8. LangGraph — why LangChain moved agents here

You just used `create_agent` from `langchain.agents`, but it's actually built
on **LangGraph**, a separate-but-related library. Worth understanding why it
exists, since "LangChain vs. LangGraph" is a genuinely common interview
question.

A chain (LCEL) is a **DAG** — a straight line or tree of steps, no loops, no
revisiting an earlier step. That's fine for RAG, but agent loops need to
*loop* ("call a tool, look at the result, maybe call another tool, maybe stop")
— LCEL alone can't express that. LangGraph models your application as an
explicit **state graph**: nodes (steps), edges (what runs next), and a shared
**state** object every node can read/update — including edges that loop back
to an earlier node, which is exactly what an agent needs.

`create_agent` is LangChain's prebuilt "graph with a reasoning-loop, an
LLM node, and a tool-calling node" — you don't have to hand-build the graph
for the common case, but you can drop down to raw LangGraph when you need
custom control flow (e.g., a human-approval step before a tool runs, or
branching between two different sub-agents).

> **Interview angle:** "LCEL chains are DAGs — no cycles. Agents need to loop
> (reason, act, observe, repeat), which needs actual graph structure with
> state and conditional edges — that's what LangGraph provides. `create_agent`
> is a prebuilt LangGraph graph for the common ReAct case; you reach for raw
> LangGraph when you need custom control flow LCEL literally cannot express."


## 9. Production concerns (the questions that separate tutorial-level from real)

Things that don't show up in a demo notebook but come up immediately in a real
system — and are common interview follow-ups once you've explained the happy
path:

- **Context window limits.** A 10-K is 100-800 pages (JPM's, from Day 15).
  You can never stuff the whole thing in a prompt — this is *why* RAG exists
  at all, not an optimization on top of "just paste the document in."
- **Cost.** Every retrieved chunk is tokens you pay for, every time. `k=3`
  vs `k=10` in a retriever is a real cost/quality tradeoff, not a free knob.
- **Latency.** Streaming (`.stream()`, Section 2) matters for perceived speed
  in a UI even when total generation time is unchanged — the user sees tokens
  immediately instead of a frozen screen.
- **Caching.** Identical questions shouldn't re-hit the LLM. LangChain has
  built-in `set_llm_cache()` for exact-match caching; semantic caching (cache
  hit on *similar*, not identical, questions) is a step further.
- **Hallucination in RAG specifically.** You saw the *good* case in
  `16Langchain-Core.ipynb` (the model correctly said "context doesn't contain
  the answer"). That only happens because the prompt explicitly instructed it
  to — remove that instruction and a weaker prompt will happily guess.
- **Observability.** LangSmith (LangChain's tracing product) logs every step
  of a chain/agent run — which chunks got retrieved, what the LLM saw, what it
  returned. Without this, debugging *why* a RAG answer was wrong means
  guessing blind.

> **Interview angle:** "The demo-to-production gap in RAG is almost always
> retrieval quality and cost control, not the LLM call itself. I'd want
> tracing (LangSmith or equivalent) from day one, an explicit 'say you don't
> know' instruction in the prompt, and to treat `k` and chunk size as tuned
> parameters, not defaults — they directly trade off cost against answer
> quality."


## 10. LangChain vs. the alternatives (a real differentiator answer)

A strong interview answer here isn't "LangChain is the best" — it's knowing
the tradeoffs:

| | LangChain | LlamaIndex | Raw SDK (`instructor`, provider SDKs) |
|---|---|---|---|
| Strength | Broad integrations, flexible composition (LCEL), agents | Deep RAG/indexing specialization, strong default retrieval quality | Full control, minimal abstraction, easiest to debug |
| Weakness | Can be over-abstracted for simple tasks; version churn (you hit this already — deprecated imports) | Less general-purpose outside RAG | You rebuild integration glue yourself |
| When to use | Multi-step pipelines, swapping providers/stores, agents | RAG-heavy apps where retrieval quality is the main lever | Single LLM calls, structured extraction (your `04email_extractor.py`), when you want zero framework overhead |

You've actually already lived this tradeoff in this curriculum: the Email
Classifier (Days 4-5) used `instructor` directly — no LangChain — because it
was one structured extraction call, and a framework would've been pure
overhead. Project 1 uses LangChain because it's a genuine multi-step pipeline
(load → split → embed → store → retrieve → generate) where the composition
actually earns its keep.

> **Interview angle:** "I don't reach for LangChain by default — for a single
> structured-extraction call I'd use `instructor` directly, like I did on my
> Email Classifier project. LangChain earns its place when there's real
> multi-step composition — RAG pipelines, agents — where LCEL's swappable
> pieces and prebuilt patterns save real integration work."


## 11. Interview Q&A cheat sheet

Quick-reference — skim this section alone the night before an interview.

**Q: What is LangChain, in one sentence?**
A framework for composing LLM applications from interchangeable pieces (models,
retrievers, tools, parsers) via a consistent interface (Runnables) and a
composition syntax (LCEL).

**Q: What is LCEL and why does `|` work across totally different components?**
Every component implements the same Runnable interface (`invoke`/`batch`/
`stream`/`ainvoke`). `|` builds a `RunnableSequence`; a dict of Runnables
becomes a `RunnableParallel`. Same interface everywhere is what makes chaining
uniform.

**Q: What's the difference between a chain and an agent?**
A chain's control flow is fixed at code-writing time. An agent's control flow
(which tools, what order) is decided by the LLM at run time, in a loop
(ReAct pattern: reason → act → observe → repeat).

**Q: Why does LangChain use LangGraph for agents instead of plain LCEL?**
LCEL chains are DAGs — no cycles. Agent loops need to revisit earlier steps
based on new information, which needs real graph structure (nodes, edges,
shared state, conditional/looping edges) — LangGraph provides that.

**Q: How would you reduce hallucination in a RAG system?**
Explicit "answer only from context, say so if you don't know" prompting
(demonstrated live in `16Langchain-Core.ipynb`), better retrieval (MMR/hybrid
search so the right chunk actually gets retrieved), and citing sources so
wrong answers are at least checkable.

**Q: Why split documents into chunks instead of embedding the whole thing?**
A single embedding is a fixed-size (e.g. 1536-number) summary of the *entire*
input — a multi-topic document dilutes into a blended, less-matchable vector
(proven with real numbers in `13Embeddings.ipynb`: 0.81 similarity for a
focused doc vs. 0.48 for the same-length doc mixed with unrelated content).

**Q: `RecursiveCharacterTextSplitter` vs `TokenTextSplitter` — when does it matter?**
Token count, not character count, determines context-window/cost limits.
`RecursiveCharacterTextSplitter` approximates this with characters (fast,
usually close enough); `TokenTextSplitter` measures the real thing when you
need precision against a hard token budget.

**Q: What does MMR trade off against plain similarity search?**
Some top-1 relevance, in exchange for diversity across the retrieved set —
avoids burning your context budget on 3 near-duplicate chunks that all say the
same thing.

**Q: What's hybrid search?**
Combining vector similarity search with keyword search (commonly BM25) — an
ensemble retriever. Handles the case embeddings alone miss: exact-match terms
like ticker symbols, legal citations, or specific figures.

**Q: When would you NOT use LangChain?**
A single structured-extraction call with no real pipeline — `instructor`
directly (as in the Email Classifier project) has less overhead and fewer
moving parts to debug when there's nothing to actually compose.

**Q: How do you give an LLM "memory" across turns?**
You don't — LLMs are stateless between calls. Every apparent memory is the
app re-sending prior conversation turns as part of the prompt
(`MessagesPlaceholder` in LangChain's case). This has direct cost/context-
window implications as a conversation grows.

**Q: What's the difference between `.with_structured_output()` and manually
parsing JSON from a text response?**
`with_structured_output` binds a schema to the call and validates the
response against it automatically (via tool-calling or JSON mode under the
hood) — same category of guarantee `instructor` gives you, just built into
LangChain's Runnable interface so it composes with `|`.

**Q: What would you monitor in a production RAG system?**
Which chunks got retrieved per query (retrieval quality), latency per step,
token cost per query, and rate of "I don't know" vs. hallucinated answers.
LangSmith traces every step of a chain/agent run for exactly this.
